# Upper Extremity Fugl-Meyer outcomes with home-based brain-computer interface therapy in routine care

Analysis code for the manuscript *A Retrospective Real-World Assessment of Post-Stroke Motor Function Improvements Using a
Home-Based Brain Computer Interface*. The outcome is the Upper Extremity Fugl-Meyer (UEFM) total score. Section 1 prepares the
data as described in the Methods; Sections 2–4 reproduce the Results, with one subsection per table, figure, or reported summary.

| Section | Manuscript |
|---|---|
| 1 | Methods: input data, patient variables, device use, follow-up weeks, study population |
| 2 | Results 3.1, patient characteristics: Figure 1B, Table 1, device use, Supplemental Tables 1–3 |
| 3 | Results 3.2, UEFM improvements after 12 weeks: Figure 2, Supplemental Figure 1, Supplemental Table 4 |
| 4 | Results 3.3, UEFM changes over one year: Supplemental Figures 2–4, Figure 3, Supplemental Figures 5–6 |

Python (pandas, NumPy, SciPy, Matplotlib) handles the data, the paired analyses, and the figures. The mixed model for repeated
measures (MMRM) and its multiple-imputation sensitivity analyses are fitted in R (`R/`, packages `mmrm` and `jsonlite`),
called through `Rscript`.

The input files are patient-level records with direct identifiers removed (dates retained) and are not distributed (see
`README.md`). Tables are written to
`outputs/` as CSV and figures to `outputs/figures/` as PNG and SVG. `outputs/` also receives the patient-level files passed to R
(`mmrm_input.csv`, `mmrm_impute_cells.csv`), so it stays as private as the input data.

## 0. Setup

In [ ]:
import json, os, platform, subprocess
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import MaxNLocator
from scipy import stats
from scipy.stats import gaussian_kde

ROOT = Path.cwd()
DATA = Path(os.environ.get("RWE_DATA_DIR", ROOT / "data"))
OUT = Path(os.environ.get("RWE_OUTPUT_DIR", ROOT / "outputs"))
FIG, RDIR = OUT / "figures", ROOT / "R"
FIG.mkdir(parents=True, exist_ok=True)

r_versions = subprocess.run(["Rscript", "-e", "cat(as.character(getRversion()), as.character(packageVersion('mmrm')))"],
                            capture_output=True, text=True, check=True).stdout.split()
print(f"Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__} | SciPy {scipy.__version__} | "
      f"Matplotlib {matplotlib.__version__} | R {r_versions[0]} | mmrm {r_versions[1]}")

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": False, "axes.edgecolor": "#0b0b0b", "axes.labelcolor": "#0b0b0b",
                     "xtick.color": "#0b0b0b", "ytick.color": "#0b0b0b", "svg.fonttype": "none"})
BLUE, ORANGE, INK, MUTED, GREY = "#2a78d6", "#eb6834", "#0b0b0b", "#898781", "#999999"
BLUE_FILL, ORANGE_FILL, GREY_FILL, BOX_FILL = "#bcd4f2", "#f7c6b3", "#e3e2dd", "#e8e8e8"


def save_fig(fig, name):
    for ext in ("png", "svg"):
        fig.savefig(FIG / f"{name}.{ext}", bbox_inches="tight")


def r1(x, digits=1):
    """Round half up for reporting (format() rounds half to even)."""
    return str(Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-digits), ROUND_HALF_UP))


def fmt_p(p):
    return "< 0.001" if p < 0.001 else f"{p:.3f}" if p < 0.1 else f"{p:.2f}"


def rscript(script, *args):
    """Run an R script from R/, print its output, and stop with R's error message if it fails."""
    run = subprocess.run(["Rscript", str(RDIR / script), *map(str, args)], capture_output=True, text=True)
    print(run.stdout.strip())
    if run.returncode != 0:
        raise RuntimeError(f"{script} failed:\n{run.stderr[-2000:]}")

## 1. Data, patient variables, and the study population

Constants used throughout. The minimal clinically important difference (MCID) is 5.25 UEFM points (Page et al., 2012);
severity bands follow Woytowicz et al. (2017).

In [ ]:
# Outcome
UEFM_MIN, UEFM_MAX = 0, 66
MCID = 5.25
SEVERITY_BANDS = [("Severe", 0, 28), ("Moderate", 29, 42), ("Mild", 43, 66)]

# Follow-up weeks: an assessment belongs to week w when it falls within w ± 2 weeks of the baseline assessment date
FOLLOW_UP_WEEKS = list(range(4, 53, 4))
HALF_WINDOW_WEEKS = 2

# Study population
RX_START = pd.Timestamp("2023-01-01")          # first prescription on or after this date
EXTRACTION_DATE = pd.Timestamp("2026-09-06")   # data extraction; first prescriptions up to this date are included
AGE_MIN = 18
CHRONIC_MIN_DAYS = 182                         # stroke at least six months before the first device session
MIN_SESSIONS_PER_WEEK = 1.0                    # included patients averaged more than one session per week
NO_EVENT = "No MDE present"                    # review verdict: no significant detrimental medical event

# Statistics
ALPHA = 0.05
SEED = 20260903
BOOT_B = 5000                                  # patient-level bootstrap replicates
M_IMPUTATIONS = 50
TIPPING_DELTAS = [k / 2 for k in range(0, 31)] # penalty on imputed values after week 12: 0, 0.5, ..., 15 points

PID, DATE, UEFM = "PATIENT_ID", "SCORE_DATE", "UEFM"

### 1.1 Input files

All files are keyed on `PATIENT_ID`, a coded patient key.

| File | Content |
|---|---|
| `rwe_uefm.csv` | One row per UEFM assessment: date, total score, and whether it is the baseline assessment |
| `rwe_patient.csv` | One row per patient with a device or a UEFM assessment: age, sex, date of stroke and its precision, affected side, Social Deprivation Index (SDI), primary payer, prescription, shipment, device-assignment, and rental dates |
| `rwe_device_sessions.csv` | One row per logged therapy session: start time and duration |
| `rwe_prescribed.csv` | One row per prescribed patient: first prescription date and whether a device was received |
| `rwe_patient_core.csv` | Dominant hand |
| `mde_review_deidentified.csv` | Clinical review of significant detrimental medical events (verdict column `MDE?`) |
| `rwe_prescribed_dropoff.csv` | Payer recorded on each prescribed patient's insurance case and case record |
| `prescribed_dropoff_reasons_2026-09-14.csv` | Reason a prescribed patient did not receive a device (`REASON_BIN`) |
| `recipient_flow_step_eeg_status_2026-09-14.csv` | Pre-therapy EEG screen result for each device recipient (`EEG_SCREEN_STATUS`) |

In [ ]:
uefm_tbl = pd.read_csv(DATA / "rwe_uefm.csv", dtype={PID: str}, parse_dates=[DATE])
patient_tbl = pd.read_csv(DATA / "rwe_patient.csv", dtype={PID: str},
                          parse_dates=["DATE_OF_STROKE", "FIRST_RX_RECEIVED_DATE", "FIRST_DEVICE_SHIP_DATE", "FIRST_SALES_ORDER_SHIP_DATE",
                                       "FIRST_DEVICE_ASSIGNED_DATE", "LAST_DEVICE_ASSOCIATION_ENDED_DATE", "LAST_RENTAL_END_DATE"]).set_index(PID)
sessions = pd.read_csv(DATA / "rwe_device_sessions.csv", dtype={PID: str}, parse_dates=["SESSION_START"])
prescribed = pd.read_csv(DATA / "rwe_prescribed.csv", dtype={PID: str}, parse_dates=["FIRST_RX_RECEIVED_DATE"])
dominant_hand = pd.read_csv(DATA / "rwe_patient_core.csv", dtype={PID: str}, usecols=[PID, "DOMINANT_HAND"]).set_index(PID).DOMINANT_HAND
event_review = pd.read_csv(DATA / "mde_review_deidentified.csv", dtype={PID: str}, usecols=[PID, "MDE?"])
case_payer = pd.read_csv(DATA / "rwe_prescribed_dropoff.csv", dtype={PID: str},
                         usecols=[PID, "PAYER_TYPE_PRIMARY_LIVE", "PAYER_TYPE_OPPORTUNITY", "AS_OF_DATE"]).set_index(PID)
assert patient_tbl.index.is_unique and prescribed[PID].is_unique
assert set(uefm_tbl[PID]) <= set(patient_tbl.index) and set(sessions[PID]) <= set(patient_tbl.index)

verdict = event_review["MDE?"].astype("string").str.strip()
event_ids = set(event_review.loc[verdict.notna() & (verdict != "") & (verdict != NO_EVENT), PID])
DATA_CUTOFF = uefm_tbl[DATE].max().normalize()          # latest assessment date
assert DATA_CUTOFF <= EXTRACTION_DATE and (pd.to_datetime(case_payer.AS_OF_DATE) == EXTRACTION_DATE).all(), "inputs do not match EXTRACTION_DATE"
print(f"data extracted {EXTRACTION_DATE.date()}; latest assessment {DATA_CUTOFF.date()}")

### 1.2 Patient variables

The baseline is the assessment recorded as baseline (one per patient). Age at baseline is the recorded age (capped at 90)
minus the whole years between the baseline assessment and the extraction date. Time since stroke is the baseline date minus
the date of stroke. The device receipt date is the earliest of the shipment, device-assignment, and delivered-order dates.

In [ ]:
scores = uefm_tbl.loc[uefm_tbl[UEFM].notna(), [PID, "SCORE_TIMELINE", DATE, UEFM]]
assert scores[UEFM].between(UEFM_MIN, UEFM_MAX).all()
baseline = scores[scores.SCORE_TIMELINE == "baseline"].set_index(PID)
assert baseline.index.is_unique

patients = pd.DataFrame(index=patient_tbl.index.sort_values())
patients["age_at_extraction"] = patient_tbl.AGE
patients["sex"] = patient_tbl.GENDER
patients["side_affected"] = patient_tbl.SIDE_AFFECTED
patients["dominant_hand"] = dominant_hand
patients["sdi"] = patient_tbl.SDI_SCORE
patients["payer"] = patient_tbl.PAYER_TYPE_PRIMARY
patients["stroke_date"] = patient_tbl.DATE_OF_STROKE
patients["stroke_date_precision"] = patient_tbl.DATE_OF_STROKE_PRECISION
patients["rx_date"] = patient_tbl.FIRST_RX_RECEIVED_DATE
delivered = patient_tbl.ANY_SALES_ORDER_DELIVERED.astype(str).str.lower() == "true"
patients["receipt_date"] = pd.concat([patient_tbl.FIRST_DEVICE_SHIP_DATE, patient_tbl.FIRST_DEVICE_ASSIGNED_DATE,
                                      patient_tbl.FIRST_SALES_ORDER_SHIP_DATE.where(delivered)], axis=1).min(axis=1)
patients["uefm_base"] = baseline[UEFM]
patients["baseline_date"] = baseline[DATE]
patients["severity"] = pd.cut(patients.uefm_base, bins=[lo - 0.5 for _, lo, _ in SEVERITY_BANDS] + [UEFM_MAX + 0.5],
                              labels=[band for band, _, _ in SEVERITY_BANDS])
years_since_baseline = np.floor((EXTRACTION_DATE - patients.baseline_date).dt.days / 365.25)
patients["age"] = (patients.age_at_extraction - years_since_baseline).where(patients.baseline_date.notna(), patients.age_at_extraction)
time_since_stroke = (patients.baseline_date - patients.stroke_date).dt.days / 365.25
patients["time_since_stroke_years"] = time_since_stroke.where(time_since_stroke >= 0)
assert patients.stroke_date_precision.dropna().isin(["day", "month", "year"]).all()

### 1.3 Device use

A session counts when its logged duration is positive. Sessions per week is the number of sessions from the baseline assessment
to the patient's most recent UEFM assessment (the latest assessment date in the data when no later assessment exists), divided by
that interval in weeks (at least one week).

In [ ]:
logged = sessions[sessions.DURATION_SECONDS > 0]
by_patient = logged.groupby(PID).SESSION_START
patients["n_sessions"] = by_patient.size().reindex(patients.index).fillna(0).astype(int)
patients["first_session"] = by_patient.min().dt.normalize()

last_assessment = scores.dropna(subset=[DATE]).groupby(PID)[DATE].max().reindex(patients.index)
use_start = patients.baseline_date.fillna(patients.first_session)
use_end = last_assessment.where(last_assessment > use_start, DATA_CUTOFF).fillna(DATA_CUTOFF)
use_weeks = ((use_end - use_start).dt.days / 7).clip(lower=1)
in_interval = logged.assign(day=logged.SESSION_START.dt.normalize()).join(pd.DataFrame({"start": use_start, "end": use_end}), on=PID)
n_in_interval = (in_interval[(in_interval.day >= in_interval.start) & (in_interval.day <= in_interval.end)]
                 .groupby(PID).size().reindex(patients.index).fillna(0))
patients["sessions_per_week"] = n_in_interval / use_weeks

### 1.4 Follow-up weeks

Weeks elapsed from the baseline assessment date to each follow-up assessment date; an assessment is assigned to week *w* when
*w* − 2 ≤ elapsed weeks < *w* + 2. Assessments before baseline, within two weeks of it, at 54 weeks or later, or without a date
are not assigned. Two assessments of one patient in the same week are averaged.

In [ ]:
def follow_up_week(elapsed_weeks):
    for w in FOLLOW_UP_WEEKS:
        if w - HALF_WINDOW_WEEKS <= elapsed_weeks < w + HALF_WINDOW_WEEKS:
            return w
    return np.nan


follow = scores[scores.SCORE_TIMELINE != "baseline"].join(patients.baseline_date, on=PID)
follow["week"] = ((follow[DATE] - follow.baseline_date).dt.days / 7).map(follow_up_week)
visits = follow.dropna(subset=["week"]).groupby([PID, "week"], as_index=False).agg(uefm=(UEFM, "mean"))
visits["week"] = visits.week.astype(int)

### 1.5 Study population

Patients with a first prescription from January 1, 2023 to the extraction date who received a device (a shipment, device
assignment, delivered order, or logged session), applied in order: age 18 or older; a baseline UEFM; a stroke at least six months
before the first device session (a date recorded only to the month or year is read at its last possible day, and a patient without
a recorded date is excluded); more than one session per week on average; no confirmed significant detrimental medical event.
Hemiparesis or hemiplegia is a condition of the program's eligibility screening and is not re-checked here.

In [ ]:
rx_window = prescribed.FIRST_RX_RECEIVED_DATE.between(RX_START, EXTRACTION_DATE)
received = prescribed.DEVICE_RECEIVED.astype(str).str.lower() == "true"

stroke_latest = patients.stroke_date.mask(patients.stroke_date_precision == "month", patients.stroke_date + pd.offsets.MonthEnd(0))
stroke_latest = stroke_latest.mask(patients.stroke_date_precision == "year", patients.stroke_date + pd.offsets.YearEnd(0))
stroke_to_first_use = (patients.first_session.fillna(patients.receipt_date) - stroke_latest).dt.days

crit = pd.DataFrame(index=patients.index)
crit["recipient"] = patients.rx_date.between(RX_START, EXTRACTION_DATE) & (patients.receipt_date.notna() | (patients.n_sessions > 0))
crit["adult"] = patients.age_at_extraction >= AGE_MIN
crit["baseline"] = patients.uefm_base.notna()
crit["chronic"] = stroke_to_first_use >= CHRONIC_MIN_DAYS              # False when the stroke date or first use is missing
crit["device_use"] = patients.sessions_per_week > MIN_SESSIONS_PER_WEEK
crit["no_event"] = ~patients.index.isin(event_ids)
assert set(patients.index[crit.recipient]) == set(prescribed.loc[rx_window & received, PID])

CRITERIA = ["adult", "baseline", "chronic", "device_use", "no_event"]
steps = {"recipient": crit.recipient}
for prev, name in zip(["recipient"] + CRITERIA, CRITERIA):
    steps[name] = steps[prev] & crit[name]
cohort = patients[steps["no_event"]]

wide = (pd.concat([pd.DataFrame({PID: cohort.index, "week": 0, "uefm": cohort.uefm_base.to_numpy()}), visits[visits[PID].isin(cohort.index)]])
        .pivot(index=PID, columns="week", values="uefm").reindex(columns=[0] + FOLLOW_UP_WEEKS))   # patients × weeks, week 0 = baseline
assert wide[0].notna().all() and wide.index.equals(cohort.index)
followed = wide.index[wide.drop(columns=0).notna().any(axis=1)]       # patients with at least one follow-up UEFM
print(f"study cohort: {len(cohort)} patients, {len(followed)} with at least one follow-up UEFM")


def is_due(baseline_date, week):
    """A follow-up week is due when it had passed by the latest assessment date."""
    return baseline_date + pd.Timedelta(days=7 * week) <= DATA_CUTOFF

## 2. Patient characteristics (Results 3.1)

### Figure 1B. Patient flow

Prescribed patients who did not initiate therapy are grouped by the documented reason they did not receive a device (payer
denial or coverage barrier; case closed for other reasons; case still open) or, for device recipients, a failed pre-therapy
EEG screen (a recorded failure with no later pass and no baseline assessment). The inclusion criteria are then applied in
order to the patients who initiated therapy; a patient who fails several criteria is counted at the first.

In [ ]:
reason = pd.read_csv(DATA / "prescribed_dropoff_reasons_2026-09-14.csv", dtype={PID: str}, usecols=[PID, "REASON_BIN"]).set_index(PID).REASON_BIN
screen = pd.read_csv(DATA / "recipient_flow_step_eeg_status_2026-09-14.csv", dtype={PID: str},
                     usecols=[PID, "EEG_SCREEN_STATUS"]).set_index(PID).EEG_SCREEN_STATUS
REASON_GROUP = {"A": "Payer/coverage barrier", "B": "Case closed", "C": "Case closed", "D": "Case closed", "F": "Case closed",
                "E": "Case open"}

rx_ids = pd.Index(prescribed.loc[rx_window, PID])
no_device = reason.reindex(prescribed.loc[rx_window & ~received, PID])
assert no_device.isin(list(REASON_GROUP)).all(), "a prescribed patient without a device lacks a recorded reason"
no_device = no_device.map(REASON_GROUP)
recipients = patients.index[steps["recipient"]]
recipient_screen = screen.reindex(recipients)
assert recipient_screen.isin(["F", "X", "P", "H", "N"]).all(), "a device recipient lacks an EEG screen status"
failed_eeg = recipients[(recipient_screen == "F").to_numpy() & ~crit.baseline.loc[recipients].to_numpy()]
initiated = recipients.difference(failed_eeg)

EXCLUSION_LABELS = {"adult": "Age unknown or under 18", "baseline": "No baseline UEFM", "chronic": "Chronicity less than 6 months or N/A",
                    "device_use": "Average device use ≤ one session/week", "no_event": "Medical exclusion"}
remaining, excluded = initiated, {}
for name in CRITERIA:
    kept = remaining[crit.loc[remaining, name].to_numpy()]
    excluded[EXCLUSION_LABELS[name]] = len(remaining) - len(kept)
    remaining = kept
assert remaining.equals(cohort.index)

reason_counts = no_device.value_counts()
not_treated = {"Payer/coverage barrier": int(reason_counts.get("Payer/coverage barrier", 0)), "Failed EEG Screen": len(failed_eeg),
               "Case closed": int(reason_counts.get("Case closed", 0)), "Case open": int(reason_counts.get("Case open", 0))}
assert sum(not_treated.values()) + len(initiated) == len(rx_ids)
flow = pd.Series({"Prescribed IpsiHand": len(rx_ids), "Not treated": sum(not_treated.values()),
                  **{f"  {k}": v for k, v in not_treated.items()}, "Initiated Therapy": len(initiated),
                  "Excluded": sum(excluded.values()), **{f"  {k}": v for k, v in excluded.items()},
                  "Real-world users": len(cohort), "Patients with follow-up UEFM": len(followed)}, name="n")
flow.to_csv(OUT / "figure1b_patient_flow.csv")
flow.to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 6.5)); ax.set_xlim(0, 7.2); ax.set_ylim(0, 6.5); ax.axis("off")


def box(x, y, w, h, entries, fill="white"):
    # entries: (count, label) pairs stacked top to bottom, separated by dashed rules
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.12", fc=fill, ec=INK, lw=1.0))
    step = h / len(entries)
    for i, (n, label) in enumerate(entries):
        top = y + h - step * i
        ax.text(x + w / 2, top - 0.07, f"{n}", ha="center", va="top", fontsize=10, fontweight="bold")
        ax.text(x + w / 2, top - 0.29, label, ha="center", va="top", fontsize=8.5, linespacing=1.1)
        if i:
            ax.plot([x + 0.25, x + w - 0.25], [top, top], color=MUTED, lw=0.7, ls=(0, (4, 3)))


def arrow(x0, y0, x1, y1):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.0))


xm, wm, xc = 0.3, 2.2, 1.4                                            # main column
main = [(5.75, 0.62, "Prescribed IpsiHand"), (4.0, 0.62, "Initiated Therapy"), (1.7, 0.62, "Real-world users"),
        (0.1, 0.8, "Patients with follow-up UEFM")]
for (y, h, key), nxt in zip(main, main[1:] + [None]):
    box(xm, y, wm, h, [(flow[key], key.replace("with follow-up", "with\nfollow-up"))])
    if nxt:
        arrow(xc, y, xc, nxt[0] + nxt[1])

SIDE_LABELS = {"Chronicity less than 6 months or N/A": "Chronicity less than 6\nmonths or N/A",
               "Average device use ≤ one session/week": "Average device use\n≤ one session/week"}
xs, ws = 4.3, 2.75                                                    # side boxes
box(xs, 4.15, ws, 2.25, [(n, label) for label, n in not_treated.items()], fill=BOX_FILL)
box(xs, 0.1, ws, 3.85, [(n, SIDE_LABELS.get(label, label)) for label, n in excluded.items()], fill=BOX_FILL)
for y_branch, total, label in [(5.2, flow["Not treated"], "Not treated"), (3.1, flow["Excluded"], "Excluded")]:
    arrow(xc, y_branch, xs, y_branch)
    ax.text(3.35, y_branch + 0.05, f"{total}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.text(3.35, y_branch - 0.05, label, ha="center", va="top", fontsize=8.5)
save_fig(fig, "figure1b_patient_flow"); plt.show()

### Table 1. Patient demographics and baseline characteristics

The study cohort and the patients with at least one follow-up UEFM. Continuous characteristics: mean (SD). Categorical
characteristics: n (% of the column); N/A counts missing or unlisted values.

In [ ]:
def table1_column(df, side_na_row):
    N = len(df)
    mean_sd = lambda x: f"{r1(x.mean())} ({r1(x.std(ddof=1))})"
    n_pct = lambda k: f"{int(k)} ({r1(100 * k / N)})"
    side = df.side_affected.where(df.side_affected.isin(["Left", "Right"]))
    sex = df.sex.where(df.sex.isin(["Male", "Female"]))
    hand = df.dominant_hand.where(df.dominant_hand.isin(["Right", "Left", "Ambidextrous"]))
    rows = [("Age at baseline, years (mean, SD)", mean_sd(df.age)), ("Sex, n (%)", "")]
    rows += [(f"  {s}", n_pct((sex == s).sum())) for s in ["Male", "Female"]] + [("  N/A", n_pct(sex.isna().sum()))]
    rows += [("Baseline UEFM, 0–66 (mean, SD)", mean_sd(df.uefm_base)), ("Baseline severity band, n (%)", "")]
    rows += [(f"  {band} ({lo}–{hi})", n_pct((df.severity == band).sum())) for band, lo, hi in SEVERITY_BANDS]
    rows += [("Time since stroke at baseline, years (mean, SD)", mean_sd(df.time_since_stroke_years)), ("Affected upper extremity, n (%)", "")]
    rows += [(f"  {s}", n_pct((side == s).sum())) for s in ["Left", "Right"]] + ([("  N/A", n_pct(side.isna().sum()))] if side_na_row else [])
    rows += [("Dominant hand, n (%)", "")] + [(f"  {h}", n_pct((hand == h).sum())) for h in ["Right", "Left", "Ambidextrous"]]
    rows += [("  N/A", n_pct(hand.isna().sum())), ("Social Deprivation Index, 1–100 (mean, SD)", mean_sd(df.sdi))]
    return rows                                                       # (row label, value); labels repeat across sections


side_na_row = not cohort.side_affected.isin(["Left", "Right"]).all()   # an N/A row only when a side is missing
full, with_followup = table1_column(cohort, side_na_row), table1_column(cohort.loc[followed], side_na_row)
table1 = pd.DataFrame({f"Full patient cohort (N={len(cohort)})": [value for _, value in full],
                       f"Patients with UEFM follow up (N={len(followed)})": [value for _, value in with_followup]},
                      index=pd.Index([label for label, _ in full], name="Characteristic"))
table1.to_csv(OUT / "table1_characteristics.csv")
tss = cohort.time_since_stroke_years
print(f"time since stroke at baseline: range {tss.min():.2f} to {tss.max():.2f} years")
for label, df in [("full cohort", cohort), ("with follow-up", cohort.loc[followed])]:
    print(f"missing values, {label}: " + ", ".join(f"{name} {int(df[col].isna().sum())}" for name, col in
          [("age", "age"), ("time since stroke", "time_since_stroke_years"), ("Social Deprivation Index", "sdi")]))
table1

### Device use

Average sessions per week in the study cohort and among the patients excluded for averaging one session per week or fewer.

In [ ]:
low_use = patients.index[steps["chronic"] & ~crit.device_use]
for label, x in [("study cohort", cohort.sessions_per_week), ("excluded for device use", patients.loc[low_use, "sessions_per_week"])]:
    print(f"{label} (n = {len(x)}): mean {r1(x.mean(), 2)} (SD {r1(x.std(ddof=1), 2)}), range {x.min():.2f} to {x.max():.2f} sessions per week")

### Supplemental Table 1. Follow-up UEFM availability

Patients with at least one follow-up UEFM, at each follow-up week: observed, due but absent (the week had passed by the latest
assessment date), or not yet due. % Observed = observed / (observed + due but absent).

In [ ]:
base_date = cohort.loc[followed, "baseline_date"]
assert base_date.notna().all()
rows = []
for w in FOLLOW_UP_WEEKS:
    observed = wide.loc[followed, w].notna()
    due = is_due(base_date, w)
    rows.append({"Follow-up week": w, "# Observed": int(observed.sum()), "# Due but absent": int((~observed & due).sum()),
                 "# Not yet due": int((~observed & ~due).sum())})
st1 = pd.DataFrame(rows).set_index("Follow-up week")
st1["% Observed"] = [r1(100 * o / (o + d)) for o, d in zip(st1["# Observed"], st1["# Due but absent"])]
st1.to_csv(OUT / "supplemental_table1_followup_availability.csv")
st1

### Supplemental Table 2. Patient primary payer type

Four populations: patients prescribed IpsiHand (first prescription in the study period), patients who initiated therapy (device
recipients aged 18 or older with at least one logged session), the study cohort, and the patients with a follow-up UEFM. The payer
is the primary payer on the patient record (a blank entry counts as unrecorded); for prescribed patients with no patient record,
the payer on the insurance case, else on the case record. Percentages are of the column total.

In [ ]:
PAYER_BUCKETS = {"Medicare": ["Medicare", "Medicare FFS"], "Medicare Advantage": ["Medicare Advantage"],
                 "Commercial / Private": ["Commercial", "Commercial or Private Insurance"], "Medicaid": ["Medicaid", "Managed Medicaid"],
                 "Other / multiple": ["TriCare", "multiple", "VA", "No insurance"]}
raw_to_bucket = {raw: bucket for bucket, raws in PAYER_BUCKETS.items() for raw in raws}

case = case_payer.reindex(rx_ids)
payer = case.PAYER_TYPE_PRIMARY_LIVE.fillna(case.PAYER_TYPE_OPPORTUNITY)
record = patients.payer.reindex(rx_ids)
both = record.notna() & case.PAYER_TYPE_PRIMARY_LIVE.notna()
assert (record[both] == case.PAYER_TYPE_PRIMARY_LIVE[both]).all()
has_record = rx_ids.isin(patients.index)
payer[has_record] = record[has_record]                                # the patient record wherever the patient has one
assert set(payer.dropna()) <= set(raw_to_bucket)
bucket = payer.map(raw_to_bucket).fillna("Unrecorded")

populations = {"Prescribed IpsiHand": rx_ids, "Initiated therapy": patients.index[steps["adult"] & (patients.n_sessions > 0)],
               "Real-world users": cohort.index, "With a follow-up UEFM": followed}
ROWS = list(PAYER_BUCKETS) + ["Unrecorded"]
columns = {}
for label, ids in populations.items():
    member_bucket = bucket.reindex(ids)
    assert member_bucket.notna().all(), f"{label}: a population member is not in the prescribed table"
    counts = member_bucket.value_counts().reindex(ROWS).fillna(0).astype(int)
    counts["Medicare or Medicare Advantage (combined)"] = counts["Medicare"] + counts["Medicare Advantage"]
    cells = {row: f"{n:,} ({r1(100 * n / len(ids))}%)" for row, n in counts.items()}
    columns[f"{label} (n = {len(ids):,})"] = {**cells, "Total": f"{len(ids):,}"}
st2 = pd.DataFrame(columns); st2.index.name = "Primary payer type"
st2.to_csv(OUT / "supplemental_table2_payer.csv")
st2

### Supplemental Table 3. Active users at each assessment window

An active user has at least one logged session within the assessment window (week ± 2 weeks from the baseline assessment date).
The denominator is the patients with a device during the window: the week had passed by the extraction date, and the device had
not been returned (or the rental ended) before the window opened. Patients without a device-ownership record count as having a
device.

In [ ]:
origin = patients.baseline_date.fillna(patients.first_session)
device_end = pd.concat([patient_tbl.LAST_DEVICE_ASSOCIATION_ENDED_DATE, patient_tbl.LAST_RENTAL_END_DATE], axis=1).max(axis=1)
still_open = (patient_tbl.N_OPEN_DEVICE_ASSOCIATIONS.fillna(0) > 0) | (patient_tbl.N_ACTIVE_RENTAL_SESSIONS.fillna(0) > 0)
device_end = device_end.mask(still_open)                              # no end date: device still held at extraction

session_days = logged.assign(day=logged.SESSION_START.dt.normalize())[[PID, "day"]].join(origin.rename("origin"), on=PID)
session_days["week"] = ((session_days.day - session_days.origin).dt.days / 7).map(follow_up_week)
used = (session_days.dropna(subset=["week"]).assign(week=lambda d: d.week.astype(int))
        .groupby([PID, "week"]).size().unstack(fill_value=0) > 0)     # patients × weeks: any session in the window


def active_users(ids):
    o, end = origin.reindex(ids), device_end.reindex(ids)
    assert o.notna().all()
    cells = {}
    for w in FOLLOW_UP_WEEKS:
        due = o + pd.Timedelta(days=7 * w) <= EXTRACTION_DATE
        has_device = due & ~(end.notna() & (end < o + pd.Timedelta(days=7 * (w - HALF_WINDOW_WEEKS))))
        active = used.reindex(index=ids, columns=[w])[w].eq(True) & has_device
        cells[w] = f"{int(active.sum())}/{int(has_device.sum())} ({r1(100 * active.sum() / has_device.sum())}%)"
    return pd.Series(cells)


st3 = pd.DataFrame({f"Real-world users, n = {len(cohort)}": active_users(cohort.index),
                    f"With a follow-up UEFM, n = {len(followed)}": active_users(followed)})
st3.index.name = "Assessment week"
st3.to_csv(OUT / "supplemental_table3_active_users.csv")
st3

## 3. UEFM improvements after 12 weeks (Results 3.2)

### Change from baseline at each follow-up week

Among study-cohort patients with both a baseline and that week's UEFM: mean change with 95% confidence intervals (t distribution,
and patient-level bootstrap), SD, median and interquartile range (IQR), Cohen's dz (mean change / SD of change), the numbers
improving (change > 0) and reaching the MCID (change ≥ 5.25), and the paired t-test, with p values adjusted across the 13 weeks
by the Benjamini–Hochberg (BH) procedure. Week 12 is the primary timepoint.

In [ ]:
rng = np.random.default_rng(SEED)


def paired_change(b, f):
    b, f = np.asarray(b, float), np.asarray(f, float)
    d = f - b; n = len(d); mean, sd = d.mean(), d.std(ddof=1); se = sd / np.sqrt(n)
    t_crit = stats.t.ppf(1 - ALPHA / 2, n - 1)
    boot = rng.choice(d, size=(BOOT_B, n), replace=True).mean(axis=1)
    return {"n": n, "mean_change": mean, "sd": sd, "ci_t_lower": mean - t_crit * se, "ci_t_upper": mean + t_crit * se,
            "ci_boot_lower": np.percentile(boot, 100 * ALPHA / 2), "ci_boot_upper": np.percentile(boot, 100 * (1 - ALPHA / 2)),
            "median": np.median(d), "q1": np.percentile(d, 25), "q3": np.percentile(d, 75), "cohen_dz": mean / sd,
            "n_improved": int((d > 0).sum()), "n_mcid": int((d >= MCID).sum()), "p": stats.ttest_rel(f, b).pvalue}


def pairs(w):
    p = wide[[0, w]].dropna()
    return p[0], p[w]


PAIRED_WEEKS = [w for w in FOLLOW_UP_WEEKS if wide[w].notna().sum() >= 2]   # weeks with at least two baseline–follow-up pairs
paired = pd.DataFrame({w: paired_change(*pairs(w)) for w in PAIRED_WEEKS}).T
paired.index.name = "week"
paired["p_bh"] = stats.false_discovery_control(paired.p)
paired.to_csv(OUT / "paired_change_by_week.csv")

w12 = paired.loc[12]; n12 = int(w12.n)
print(f"week 12 (n = {n12}): mean change {r1(w12.mean_change)}, 95% CI {r1(w12.ci_t_lower)} to {r1(w12.ci_t_upper)} (t), "
      f"{r1(w12.ci_boot_lower)} to {r1(w12.ci_boot_upper)} (bootstrap); SD {r1(w12.sd)}; median {r1(w12['median'])} "
      f"[IQR {r1(w12.q1)}, {r1(w12.q3)}]; dz {r1(w12.cohen_dz, 2)}; paired t-test p {fmt_p(w12.p)}")
print(f"improved {int(w12.n_improved)} ({r1(100 * w12.n_improved / n12)}%), did not improve {n12 - int(w12.n_improved)} "
      f"({r1(100 * (n12 - w12.n_improved) / n12)}%), reached the MCID {int(w12.n_mcid)} ({r1(100 * w12.n_mcid / n12)}%)")

### Figure 2. Changes in UEFM after 12 weeks

(A) Paired baseline and week-12 UEFM: half-violin densities, every score as a point, one line per patient, horizontal bars at
the means, and the paired t-test (*** p < 0.001). (B) Week-12 change, one bar per integer point, by category: no improvement
(change ≤ 0), improvement below the MCID, improvement ≥ MCID. The same layout serves weeks 16 and 24 (Supplemental Figures 3
and 4).

In [ ]:
def half_violin(ax, data, x0, side, edge, fill, width=0.34):
    """One side of a kernel density at x0, scaled so its widest point spans `width`."""
    y = np.linspace(max(UEFM_MIN, data.min() - 2), min(UEFM_MAX, data.max() + 2), 200)
    dens = gaussian_kde(data)(y); dens = dens / dens.max() * width
    ax.fill_betweenx(y, x0, x0 + (dens if side == "right" else -dens), facecolor=fill, edgecolor=edge, linewidth=1.5, alpha=0.9, zorder=2)


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


def paired_figure(week, name):
    b, f = (s.to_numpy(float) for s in pairs(week)); d = pd.Series(f - b)
    fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5.8), gridspec_kw={"width_ratios": [1, 1.15], "wspace": 0.28})
    xb, xf = 0.07, 0.93                                               # point strips: every score at its true value
    half_violin(axA, b, 0.0, "left", BLUE, BLUE_FILL); half_violin(axA, f, 1.0, "right", ORANGE, ORANGE_FILL)
    for yb, yf in zip(b, f):
        axA.plot([xb, xf], [yb, yf], color=GREY, alpha=0.10, linewidth=0.6, zorder=1)
    axA.scatter(np.full_like(b, xb), b, s=16, color=BLUE, edgecolor="white", lw=0.3, alpha=0.6, zorder=3)
    axA.scatter(np.full_like(f, xf), f, s=16, color=ORANGE, edgecolor="white", lw=0.3, alpha=0.6, zorder=3)
    axA.plot([-0.38, xb + 0.05], [b.mean()] * 2, color=BLUE, lw=2.6, zorder=4); axA.plot([xf - 0.05, 1.38], [f.mean()] * 2, color=ORANGE, lw=2.6, zorder=4)
    axA.plot([0, 0, 1, 1], [68.4, 70, 70, 68.4], color=INK, lw=1.0, solid_capstyle="butt", zorder=5)
    axA.text(0.5, 70.6, stars(paired.loc[week, "p"]), ha="center", va="bottom", fontsize=12)
    axA.set_xticks([0, 1]); axA.set_xticklabels(["Baseline", f"Week {week}"], fontsize=12)
    axA.set_ylabel("UEFM Total Score", fontsize=12); axA.set_ylim(-3, 76); axA.set_yticks(range(0, 61, 10)); axA.set_xlim(-0.6, 1.6)

    cats = [d[d <= 0], d[(d > 0) & (d < MCID)], d[d >= MCID]]
    pcts = [100 * len(c) / len(d) for c in cats]
    labels = [f"No improvement ({r1(pcts[0])}%)", f"Improvement below MCID ({r1(pcts[1])}%)", f"Improvement ≥ MCID ({r1(pcts[2])}%)"]
    edges = np.arange(np.floor(d.min()) - 0.5, np.ceil(d.max()) + 1.5, 1)
    axB.hist(cats, bins=edges, stacked=True, color=[MUTED, ORANGE_FILL, ORANGE], edgecolor="white", linewidth=0.6, label=labels)
    tallest = int(np.histogram(d, bins=edges)[0].max()); line_top = tallest * 1.06
    axB.plot([0, 0], [0, line_top], color=INK, lw=1.2); axB.plot([MCID, MCID], [0, line_top], color=INK, lw=1.2, ls="--")
    axB.set_ylim(0, tallest * 1.5)
    axB.annotate("No change", xy=(0, line_top), xytext=(0, tallest * 1.30), ha="center", va="bottom", fontsize=10,
                 arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.0, shrinkA=0, shrinkB=1, mutation_scale=9))
    axB.text(MCID + 0.5, line_top, f"MCID {MCID}", fontsize=10, va="bottom", ha="left")
    axB.set_xlim(min(edges[0], -5.5), max(edges[-1], 27.5) + 0.5); axB.yaxis.set_major_locator(MaxNLocator(integer=True))
    axB.set_xlabel(f"{week} Week UEFM Change", fontsize=12); axB.set_ylabel("Patients", fontsize=12)
    axB.legend(frameon=False, fontsize=9.5, loc="upper right")
    for ax, letter in ((axA, "A"), (axB, "B")):
        ax.text(-0.14, 1.04, letter, transform=ax.transAxes, fontsize=16, fontweight="bold", va="bottom", ha="left")
    save_fig(fig, name); plt.show()


paired_figure(12, "figure2_week12")

### Supplemental Figure 1. Baseline characteristics by MCID response at weeks 12, 16, and 24

At each week, the patients with both a baseline and that week's UEFM are split by whether the change reached the MCID. Age at
baseline, baseline UEFM, time since stroke, and SDI are compared by the two-sided Wilcoxon rank-sum test; q is the BH-adjusted p
across the four characteristics within the week. Medians [IQR] are those reported in Results 3.2. Boxes span the IQR with the
median as a line; whiskers reach the most extreme values within 1.5 × IQR.

In [ ]:
RESPONDER_WEEKS = [12, 16, 24]
CHARACTERISTICS = {"age": ("Age at baseline", "Years"), "uefm_base": ("Baseline UEFM", "UEFM (0–66)"),
                   "time_since_stroke_years": ("Time since stroke", "Years"), "sdi": ("Social Deprivation Index", "SDI (1–100)")}


def median_iqr(x):
    return f"{r1(x.median())} [{r1(x.quantile(.25))}, {r1(x.quantile(.75))}]"


def p_eq(p, name="p"):
    s = fmt_p(p)
    return f"{name} {s}" if s.startswith("<") else f"{name} = {s}"


rows, groups = [], {}
for w in RESPONDER_WEEKS:
    b, f = pairs(w)
    g = cohort.loc[b.index].assign(reached=(f - b) >= MCID)
    groups[w] = g
    for var, (name, _) in CHARACTERISTICS.items():
        below, above = g.loc[~g.reached, var].dropna(), g.loc[g.reached, var].dropna()
        rows.append({"week": w, "variable": var, "characteristic": name, "n < MCID": len(below), "< MCID, median [IQR]": median_iqr(below),
                     "n ≥ MCID": len(above), "≥ MCID, median [IQR]": median_iqr(above),
                     "p": stats.mannwhitneyu(above, below, alternative="two-sided").pvalue})
mcid_baseline = pd.DataFrame(rows)
mcid_baseline["q"] = mcid_baseline.groupby("week").p.transform(stats.false_discovery_control)
mcid_baseline.drop(columns="variable").to_csv(OUT / "supplemental_figure1_mcid_baseline.csv", index=False)

fig = plt.figure(figsize=(3.0 * len(CHARACTERISTICS), 3.6 * len(RESPONDER_WEEKS)), layout="constrained")
for subfig, w in zip(fig.subfigures(len(RESPONDER_WEEKS), 1), RESPONDER_WEEKS):
    g = groups[w]
    subfig.suptitle(f"Week {w}", x=0.01, ha="left", fontsize=11, fontweight="bold")
    for ax, (var, (name, unit)) in zip(subfig.subplots(1, len(CHARACTERISTICS)), CHARACTERISTICS.items()):
        s = mcid_baseline[(mcid_baseline.week == w) & (mcid_baseline.variable == var)].iloc[0]
        data = [g.loc[~g.reached, var].dropna(), g.loc[g.reached, var].dropna()]
        bp = ax.boxplot(data, positions=[0, 1], widths=0.55, patch_artist=True, medianprops=dict(color=INK, lw=1.6),
                        whiskerprops=dict(color=INK, lw=1.0), capprops=dict(color=INK, lw=1.0),
                        flierprops=dict(marker="o", markersize=3, markerfacecolor="none", markeredgecolor=MUTED))
        for patch, face, edge in zip(bp["boxes"], [GREY_FILL, ORANGE_FILL], [MUTED, ORANGE]):
            patch.set(facecolor=face, edgecolor=edge, linewidth=1.2)
        ax.set_box_aspect(1); ax.set_xlim(-0.6, 1.6)
        ax.set_xticks([0, 1], [f"< MCID\nn = {len(data[0])}", f"≥ MCID\nn = {len(data[1])}"])
        ax.set_ylabel(unit); ax.set_title(f"{name}\n{p_eq(s.p)}; {p_eq(s.q, 'q')}", fontsize=10)
save_fig(fig, "supplemental_figure1_mcid_baseline"); plt.show()
mcid_baseline.drop(columns="variable").assign(p=lambda t: t.p.map(fmt_p), q=lambda t: t.q.map(fmt_p)).set_index(["week", "characteristic"])

### Supplemental Table 4. Baseline characteristics with and without follow-up assessments

Study-cohort patients compared on four baseline characteristics (median [IQR]): those with a week-12 UEFM against those whose
week-12 assessment had come due by the latest assessment date without being recorded, and those with at least one follow-up UEFM
against those with a baseline only. Two-sided Wilcoxon rank-sum test; BH adjustment across the four characteristics within each
comparison; n is shown where a characteristic is missing for some patients.

In [ ]:
COMPARED = [("Age at baseline, years", "age"), ("Baseline UEFM, 0–66", "uefm_base"),
            ("Time since stroke at baseline, years", "time_since_stroke_years"), ("Social Deprivation Index, 1–100", "sdi")]
week12_due = is_due(cohort.baseline_date, 12)
splits = {"Week-12 UEFM assessment": (wide[12].notna(), week12_due),
          "Any follow-up UEFM assessment": (pd.Series(cohort.index.isin(followed), index=cohort.index), pd.Series(True, index=cohort.index))}


def median_n(x, N):
    x = x.dropna()
    return median_iqr(x) + (f" (n = {len(x)})" if len(x) < N else "")


blocks = []
for label, (seen, eligible) in splits.items():
    a, b = cohort[seen], cohort[~seen & eligible]
    t = pd.DataFrame([(name, median_n(a[var], len(a)), median_n(b[var], len(b)),
                       stats.mannwhitneyu(a[var].dropna(), b[var].dropna(), alternative="two-sided").pvalue) for name, var in COMPARED],
                     columns=["characteristic", "With (median, IQR)", "Without (median, IQR)", "p"])
    t["BH adjusted p"] = stats.false_discovery_control(t.p)
    t.insert(0, "comparison", f"{label} (with, n = {len(a)}; without, n = {len(b)})")
    blocks.append(t)
st4 = pd.concat(blocks).set_index(["comparison", "characteristic"])
st4.to_csv(OUT / "supplemental_table4_followup_comparison.csv")
st4.assign(p=st4.p.map(fmt_p), **{"BH adjusted p": st4["BH adjusted p"].map(fmt_p)})

## 4. UEFM changes over one year (Results 3.3)

### Observed change at each follow-up week

Mean (SD) change from baseline at every follow-up week, from the paired analysis in Section 3, with BH-adjusted paired t-test
p values.

In [ ]:
pd.DataFrame({"n": paired.n.astype(int), "mean change (SD)": [f"{r1(m)} ({r1(s)})" for m, s in zip(paired.mean_change, paired.sd)],
              "95% CI, t": [f"{r1(a)} to {r1(b)}" for a, b in zip(paired.ci_t_lower, paired.ci_t_upper)],
              "95% CI, bootstrap": [f"{r1(a)} to {r1(b)}" for a, b in zip(paired.ci_boot_lower, paired.ci_boot_upper)],
              "BH adjusted p": paired.p_bh.map(fmt_p)})

### Supplemental Figure 2. Observed changes in UEFM

Change from baseline at each follow-up week: a half-violin kernel density (left), one point per patient (right), and a horizontal
bar at the mean; the number of patients observed is printed above each week. The dashed line marks the MCID.

In [ ]:
def raincloud_violin(ax, data, x0, width, edge, fill, alpha):
    """Left-facing half-violin at x0, clipped to the observed range; skipped when n < 2 or the range is zero."""
    v = np.asarray(data, float); v = v[~np.isnan(v)]
    if v.size < 2 or np.ptp(v) == 0:
        return
    y = np.linspace(v.min(), v.max(), 200)
    dens = gaussian_kde(v)(y); dens = dens / dens.max() * width
    ax.fill_betweenx(y, x0 - dens, x0, facecolor=fill, edgecolor=edge, linewidth=0.6, alpha=alpha, zorder=2)


deltas = [f - b for b, f in map(pairs, PAIRED_WEEKS)]
jitter_rng = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(6.5, 5.6))
ax.axhline(MCID, color=INK, lw=0.9, ls=(0, (4, 2)), zorder=1)
body_w, strip_x, jitter = 1.70, 0.72, 0.25
for w, d in zip(PAIRED_WEEKS, deltas):
    raincloud_violin(ax, d, w, body_w, BLUE, BLUE_FILL, 0.85)
    ax.scatter(w + strip_x + jitter_rng.uniform(-jitter, jitter, len(d)), d, s=3.4, color=BLUE, edgecolor="none", alpha=0.38, zorder=3)
    ax.plot([w - body_w, w + strip_x + jitter], [d.mean()] * 2, color=BLUE, lw=2.0, solid_capstyle="butt", zorder=6)
lo, hi = min(d.min() for d in deltas), max(d.max() for d in deltas)
span = hi - lo; top = hi + 0.11 * span
ax.set_ylim(lo - 0.05 * span, top)
for w, d in zip(PAIRED_WEEKS, deltas):
    ax.text(w, top - 0.012 * span, f"{len(d)}", ha="center", va="top", fontsize=7)
ax.set(xlabel="Weeks", ylabel="Change in UEFM from baseline (points)", xticks=PAIRED_WEEKS)
ax.set_xlim(PAIRED_WEEKS[0] - 3, PAIRED_WEEKS[-1] + 3)
ax.legend(handles=[Line2D([], [], color=BLUE, lw=2.0, label="mean"), Line2D([], [], color=INK, lw=0.9, ls=(0, (4, 2)), label=f"MCID {MCID}")],
          frameon=False, fontsize=7.5, ncol=2, loc="lower left", bbox_to_anchor=(-0.005, 1.01), handlelength=1.2,
          columnspacing=1.2, handletextpad=0.5, borderaxespad=0.0)
fig.tight_layout(); save_fig(fig, "supplemental_figure2_observed_change"); plt.show()

### Supplemental Figures 3 and 4. Changes in UEFM after 16 and 24 weeks

Figure 2's layout for the patients with both a baseline and a week-16 (Supplemental Figure 3) or week-24 (Supplemental Figure 4)
UEFM.

In [ ]:
paired_figure(16, "supplemental_figure3_week16")
paired_figure(24, "supplemental_figure4_week24")

### Figure 3. Change in UEFM over one year (MMRM)

For patient *i* at follow-up week *t*, $Y_{it} = \mu_t + \beta\,(Y_{i0} - \bar{Y}_0) + e_{it}$: week as an unordered categorical
effect, baseline UEFM (centred at the analysis-set mean) as a covariate, and no random effects. The within-patient residuals are
multivariate normal; the covariance structure is the first in `R/mmrm_functions.R` (unstructured, then heterogeneous Toeplitz,
then heterogeneous first-order ante-dependence, ...) whose estimate converges and is positive definite. Restricted maximum
likelihood, Kenward–Roger degrees of freedom; missing assessments are handled by direct likelihood under missing at random.
The analysis set is the patients with at least one follow-up UEFM. The change from baseline at week *t* is the least-squares
(LS) mean at the mean baseline minus the mean baseline.

In [ ]:
mmrm_input = (visits[visits[PID].isin(followed)]
              .assign(pid=lambda d: d[PID], visit=lambda d: "week_" + d.week.astype(str))
              .join(cohort.uefm_base, on=PID)[["pid", "visit", "uefm", "uefm_base"]])
mmrm_input.to_csv(OUT / "mmrm_input.csv", index=False)
rscript("mmrm_fit.R", OUT / "mmrm_input.csv", OUT / "mmrm")
mm = json.load(open(OUT / "mmrm_summary.json"))
assert mm["n_obs"] == len(mmrm_input) and mm["n_patients"] == len(followed)

lsm = pd.read_csv(OUT / "mmrm_lsmeans.csv").set_index("week")
beta = pd.read_csv(OUT / "mmrm_baseline_coef.csv").iloc[0]
print(f"baseline coefficient {r1(beta.estimate, 3)} (95% CI {r1(beta.lower, 3)} to {r1(beta.upper, 3)})")
pd.DataFrame({"n": lsm.n_obs, "LS-mean change (95% CI)": [f"{r1(m)} ({r1(a)} to {r1(b)})" for m, a, b in zip(lsm.change, lsm.change_lower, lsm.change_upper)]})

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.9))
ax.plot([FOLLOW_UP_WEEKS[0] - 0.7, FOLLOW_UP_WEEKS[-2] + 0.7], [MCID, MCID], color="#7f7f7f", lw=1.3, ls=(0, (6, 4)), zorder=1)
ax.text(FOLLOW_UP_WEEKS[0] - 0.7, MCID + 0.12, "MCID", color="#7f7f7f", fontsize=12, va="bottom", ha="left")
ax.errorbar(lsm.index, lsm.change, yerr=[lsm.change - lsm.change_lower, lsm.change_upper - lsm.change], color=BLUE, lw=2.0,
            elinewidth=2.0, capsize=0, marker="o", ms=10, markeredgecolor="white", markeredgewidth=1.2, zorder=3)
ax.set_xticks(FOLLOW_UP_WEEKS[::2]); ax.set_ylim(bottom=min(-0.25, lsm.change_lower.min() - 0.25))
ax.set_xlabel("Week", fontsize=13); ax.set_ylabel("Modeled change in UEFM", fontsize=13); ax.tick_params(labelsize=12)
fig.tight_layout(); save_fig(fig, "figure3_mmrm_trajectory"); plt.show()

### Supplemental Figure 5. Carry-forward multiple imputation

Each assessment that was due but absent in the MMRM analysis set (Supplemental Table 1) is imputed as the patient's last
observed UEFM plus a draw from a normal distribution with mean zero and the model's week-specific standard deviation. The MMRM
is refitted on each of 50 completed datasets and the LS-mean changes are pooled by Rubin's rules.

In [ ]:
cells = []
for pid, row in wide.loc[followed].iterrows():
    have = row.dropna()                                               # observed weeks, baseline included as week 0
    for w in FOLLOW_UP_WEEKS:
        if w in have.index or not is_due(cohort.at[pid, "baseline_date"], w):
            continue
        cells.append({"pid": pid, "visit": f"week_{w}", "week": w, "post12": int(w > 12),
                      "last_obs_uefm": have[have.index[have.index < w].max()], "uefm_base": have[0]})
cells = pd.DataFrame(cells)
assert len(cells) == st1["# Due but absent"].sum()
cells.to_csv(OUT / "mmrm_impute_cells.csv", index=False)


def run_mi(imputation, estimate, deltas, out):
    rscript("mmrm_mi.R", f"observed={OUT / 'mmrm_input.csv'}", f"cells={OUT / 'mmrm_impute_cells.csv'}",
            f"primary={OUT / 'mmrm_summary.json'}", f"imputation={imputation}", f"estimate={estimate}", f"M={M_IMPUTATIONS}",
            f"seed={SEED}", f"deltas={','.join(map(str, deltas))}", f"out={out}")
    result = pd.read_csv(out)
    if (result.m_used < M_IMPUTATIONS).any():
        print(f"note: some completed-dataset fits failed; estimates pooled over {sorted(result.m_used.unique())} imputations")
    return result


def report_change(label, row):
    print(f"{label}: LS-mean change {r1(row.estimate)} (95% CI {r1(row.lower)} to {r1(row.upper)})")


carry_forward = run_mi("locf", "trajectory", [0], OUT / "mi_carry_forward_lsmeans.csv").set_index("week")
report_change("week 12", carry_forward.loc[12])
reached = carry_forward.index[carry_forward.estimate >= MCID]
if len(reached):
    report_change(f"first week at or above the MCID (week {reached.min()})", carry_forward.loc[reached.min()])
else:
    print("the LS-mean change stays below the MCID through week 52")
report_change("week 52", carry_forward.loc[52])

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.axhline(0, color=INK, lw=1)
mcid_line = ax.axhline(MCID, color=INK, lw=1, ls=(0, (4, 2)))
eb = ax.errorbar(carry_forward.index, carry_forward.estimate, yerr=[carry_forward.estimate - carry_forward.lower, carry_forward.upper - carry_forward.estimate],
                 color=BLUE, lw=1.2, elinewidth=1.2, capsize=0, marker="o", ms=7, markeredgecolor="white", markeredgewidth=0.8)
ax.legend([eb, mcid_line], ["Carry-forward MI LS-mean change (95% CI)", f"MCID {MCID}"], loc="upper left", frameon=False)
ax.set(xlabel="Weeks", ylabel="Change in UEFM from baseline (points)", xticks=FOLLOW_UP_WEEKS)
fig.tight_layout(); save_fig(fig, "supplemental_figure5_carry_forward_mi"); plt.show()

### Supplemental Figure 6. Tipping-point analysis

Each assessment that was due but absent is imputed from the MMRM's conditional distribution given the patient's observed scores
(missing at random); a penalty δ is subtracted from every imputed value after week 12 and the model is refitted (50 imputations,
Rubin's rules). δ increases by 0.5 points until the 95% confidence interval of the week-52 minus week-12 difference includes 0
(the tipping point), and the sweep continues one point beyond it.

In [ ]:
tipping = run_mi("mar", "contrast", TIPPING_DELTAS, OUT / "tipping_point.csv").set_index("delta")
crossed = tipping.index[(tipping.index > 0) & (tipping.lower <= 0)]          # the rule R/mmrm_mi.R uses to stop the sweep
if len(crossed):
    tipping_delta = crossed.min()
    before = tipping.loc[tipping.index[tipping.index < tipping_delta].max()]
    print(f"δ = {before.name:g}: week-52 minus week-12 {r1(before.estimate)} (95% CI {r1(before.lower)} to {r1(before.upper)})")
    print(f"tipping point: δ = {tipping_delta:g} (95% CI {r1(tipping.loc[tipping_delta, 'lower'])} to {r1(tipping.loc[tipping_delta, 'upper'])})")
else:
    tipping_delta = None
    print(f"no tipping point: the 95% CI excludes 0 at every δ up to {tipping.index.max():g}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.axhline(0, color=INK, lw=1)
ax.fill_between(tipping.index, tipping.lower, tipping.upper, color=BLUE, alpha=0.12, linewidth=0)
ax.plot(tipping.index, tipping.estimate, color=BLUE, lw=2, marker="o", ms=6)
if tipping_delta is not None:
    ax.axvline(tipping_delta, color=MUTED, lw=1)
    ax.text(tipping_delta - 0.05, ax.get_ylim()[1] * 0.97, f"tipping δ = {tipping_delta:g}", color=MUTED, fontsize=8, va="top", ha="right")
ax.set(xlabel="Penalty δ subtracted from imputed post-12-week values (points)", ylabel="One-year minus 12-week LS means (points)",
       xlim=(-0.2, tipping.index.max() + 0.2))
if len(tipping) <= 12:
    ax.set_xticks(list(tipping.index))
fig.tight_layout(); save_fig(fig, "supplemental_figure6_tipping_point"); plt.show()

### Exploratory analysis. Change from baseline at weeks 12, 16, and 24 for a baseline UEFM of 20 to 50

The paired analysis of Section 3 at weeks 12, 16, and 24 among the study-cohort patients whose baseline UEFM was 20 to 50 inclusive,
the inclusion range of earlier clinical trials of upper extremity motor recovery after stroke (Methods 2.6). Each week's patients are
those with both a baseline and that week's UEFM, so the sets differ between weeks; paired t-test p values are adjusted across the
three weeks by the BH procedure. The bootstrap generator is reseeded so these intervals do not depend on the cells above.

In [ ]:
INCLUSION_RANGE = (20, 50)                     # baseline UEFM, inclusive at both ends
RANGE_WEEKS = [12, 16, 24]
in_range = wide[0].between(*INCLUSION_RANGE)
rng = np.random.default_rng(SEED)              # reseeded so these intervals do not depend on the cells above


def range_pairs(w):
    p = wide.loc[in_range, [0, w]].dropna()
    return p[0], p[w]


range_change = pd.DataFrame({w: paired_change(*range_pairs(w)) for w in RANGE_WEEKS}).T
range_change.index.name = "week"
range_change["baseline_mean"] = [range_pairs(w)[0].mean() for w in RANGE_WEEKS]
range_change["p_bh"] = stats.false_discovery_control(range_change.p)
range_change.to_csv(OUT / "paired_change_baseline_20_50.csv")

print(f"baseline UEFM {INCLUSION_RANGE[0]} to {INCLUSION_RANGE[1]}: {int(in_range.sum())} of {len(wide)} patients, "
      f"mean baseline {r1(wide.loc[in_range, 0].mean())} (all patients {r1(wide[0].mean())})")
pd.DataFrame({"n": range_change.n.astype(int), "mean baseline": range_change.baseline_mean.map(r1),
              "mean change (SD)": [f"{r1(m)} ({r1(s)})" for m, s in zip(range_change.mean_change, range_change.sd)],
              "95% CI, t": [f"{r1(a)} to {r1(b)}" for a, b in zip(range_change.ci_t_lower, range_change.ci_t_upper)],
              "95% CI, bootstrap": [f"{r1(a)} to {r1(b)}" for a, b in zip(range_change.ci_boot_lower, range_change.ci_boot_upper)],
              "median [IQR]": [f"{r1(m)} [{r1(a)}, {r1(b)}]" for m, a, b in zip(range_change["median"], range_change.q1, range_change.q3)],
              "dz": range_change.cohen_dz.map(lambda v: r1(v, 2)),
              "improved": [f"{int(k)} ({r1(100 * k / n)}%)" for k, n in zip(range_change.n_improved, range_change.n)],
              "reached the MCID": [f"{int(k)} ({r1(100 * k / n)}%)" for k, n in zip(range_change.n_mcid, range_change.n)],
              "BH adjusted p": range_change.p_bh.map(fmt_p)})